 # Medicine Price & Generic Alternative Finder

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re


In [ ]:
df = pd.read_csv("../data/original.csv")


 ### Exploring the Dataset

In [ ]:
df.head()


In [ ]:
df.shape


In [ ]:
df.columns


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe()


In [ ]:
df.sample(5)


 ### Observations:



 - Dataset size: 253973

 - Max Price: 436000

 - Min Price: 0

 - Missing values:

   - pack_size:  22330

   - pack_unit:  22330

   - primary_strength:  25198

 ### Analysing the Data

In [ ]:
df["manufacturer"].value_counts()


In [ ]:
df["price_inr"].describe()


In [ ]:
df[df["price_inr"] == df["price_inr"].max()]


In [ ]:
df[df["price_inr"] == df["price_inr"].min()]


In [ ]:
df["active_ingredients"].value_counts().head(10)


In [ ]:
plt.figure(figsize=(8,2))

plt.boxplot(df["price_inr"], orientation="horizontal")

plt.title("Medicine Price Distribution")

plt.show()


In [ ]:
top = df["manufacturer"].value_counts().head(10)
top.plot(kind="bar")
plt.xlabel("Manufacturer")
plt.ylabel("Count")
plt.title("Top 10 Manufacturers")
plt.show()



In [ ]:
df["price_inr"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])


In [ ]:
df.nlargest(50, "price_inr")[["brand_name", "price_inr", "manufacturer"]]


 ### Observations:



 - Most common manufacturer: Sun Pharma

 - Most common active ingredient: Aceclophenac (Paracetamol)

 - Average medicine price: 270

 - 99% medicines are priced below 2899

 - Expensive outliers: All of them are justified

 ### Data Cleaning

In [ ]:
clean_df = df.copy()


In [ ]:
clean_df = clean_df.drop(columns=["product_id", "packaging_raw", "manufacturer_raw"])


In [ ]:
clean_df.duplicated().sum()


In [ ]:
clean_df = clean_df.drop_duplicates()


In [ ]:
clean_df.duplicated().sum()


In [ ]:
clean_df[clean_df["price_inr"] == 0]


In [ ]:
clean_df[["pack_size", "pack_unit", "primary_strength"]].head(10)


In [ ]:
clean_df["active_ingredients"].sample(20, random_state=42)


 ### Cleaning Notes



 - Duplicate rows: 9 dropped

 - Missing values to handle: pack_size, pack_unit, primary_strength

 - Dropped Unimportant columns: product_id, packaging_raw, manufacturer_raw

 - Medicines priced 0 are free government vaccines

In [ ]:
clean_df["manufacturer"].nunique()


In [ ]:
clean_df["manufacturer"].sample(20, random_state=42)


In [ ]:
clean_df["active_ingredients"].sample(20, random_state=42)


In [ ]:
clean_df["active_ingredients"].iloc[0]


 - Active Ingredients Column is a string representation of a Python list and is already structured.

In [ ]:
clean_df["num_active_ingredients"].value_counts()


In [ ]:
import ast

def extract_ingredient_names(text):
    if pd.isna(text):
        return None

    ingredients = ast.literal_eval(text)
    names = []

    for ingredient in ingredients:
        names.append(ingredient["name"].lower().strip())

    return names


In [ ]:
import re

def normalize_strength(value):
    if not value:
        return ""

    value = value.strip().lower()

    match = re.match(r"([\d.]+)\s*(mg|g|gm|ml)", value)

    if not match:
        return value

    number = float(match.group(1))
    unit = match.group(2)

    if unit in ("g", "gm"):
        return f"{number * 1000:.0f}mg"

    return value


In [ ]:
import ast

def create_cleaned_composition(value):
    if pd.isna(value):
        return None

    ingredients = ast.literal_eval(value)

    composition = []

    for item in ingredients:
        name = (item.get("name") or "").strip().lower()
        strength = normalize_strength(item.get("strength"))

        if strength:
            composition.append(f"{name} {strength}")
        else:
            composition.append(name)

    composition.sort()

    return " | ".join(composition)


In [ ]:
clean_df["ingredient_names"] = clean_df["active_ingredients"].apply(extract_ingredient_names)


In [ ]:
clean_df["cleaned_composition"] = clean_df["active_ingredients"].apply(create_cleaned_composition)


In [ ]:
clean_df[["active_ingredients", "ingredient_names"]].head()


In [ ]:
clean_df[["active_ingredients", "cleaned_composition"]].head()


In [ ]:
clean_df["cleaned_composition"].value_counts().head(20)


In [ ]:
clean_df["ingredient_names"].isna().sum()


In [ ]:
clean_df["therapeutic_class"].value_counts().head(20)


In [ ]:
clean_df["therapeutic_class"].isna().sum()


In [ ]:
clean_df.to_csv("../data/cleaned.csv", index=False)


 # Summary



 ## Cleaning Performed



 - Created a working copy of the dataset.

 - Removed unnecessary columns (`product_id`, `packaging_raw`, `manufacturer_raw`).

 - Verified that there were no duplicate rows.

 - Investigated missing values and decided not to remove affected rows.

 - Extracted ingredient names from the structured `active_ingredients` column into a new `ingredient_names` column.

 - Created a `cleaned_composition` column with most units converted to mg.

 - Verified that the extracted column contained no missing values.

 - Saved the cleaned dataset as `cleaned_data.csv`.

 # Feature Engineering

In [ ]:
features = [
    "manufacturer",
    "dosage_form",
    "pack_size",
    "pack_unit",
    "num_active_ingredients",
    "therapeutic_class",
    "primary_strength",
    "ingredient_names"
]

target = "price_inr"


In [ ]:
X = clean_df[features]
y = np.log1p(clean_df[target])


In [ ]:
X.head()


In [ ]:
y.head()


In [ ]:
X.dtypes


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
top_manufacturers = (
    X_train["manufacturer"]
    .value_counts()
    .nlargest(100)
    .index
)

X_train = X_train.copy()
X_test = X_test.copy()

X_train["manufacturer"] = X_train["manufacturer"].apply(
    lambda x: x if x in top_manufacturers else "Other"
)

X_test["manufacturer"] = X_test["manufacturer"].apply(
    lambda x: x if x in top_manufacturers else "Other"
)


In [ ]:
print(X_train["manufacturer"].nunique())
print(X_test["manufacturer"].nunique())


In [ ]:
categorical_features = [
    "manufacturer",
    "dosage_form",
    "pack_unit",
    "therapeutic_class"
]

numerical_features = [
    "pack_size",
    "num_active_ingredients"
]

ingredient_feature = "ingredient_names"


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[categorical_features])

X_train_encoded = encoder.transform(X_train[categorical_features])

X_test_encoded = encoder.transform(X_test[categorical_features])


In [ ]:
encoded_train_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
encoded_test_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)
encoded_train_df.shape


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()

X_train_ingredients = mlb.fit_transform(
    X_train["ingredient_names"]
)

X_test_ingredients = mlb.transform(
    X_test["ingredient_names"]
)


In [ ]:
ingredient_train_df = pd.DataFrame(
    X_train_ingredients,
    columns=mlb.classes_,
    index=X_train.index
)

ingredient_test_df = pd.DataFrame(
    X_test_ingredients,
    columns=mlb.classes_,
    index=X_test.index
)


In [ ]:
def extract_strength(value):
    if pd.isna(value):
        return None

    value = str(value)

    match = re.search(r"\d+\.?\d*", value)

    if match:
        return float(match.group())

    return None


In [ ]:
X_train_strength = X_train["primary_strength"].apply(extract_strength)

X_test_strength = X_test["primary_strength"].apply(extract_strength)

X_train[
    ["primary_strength"]
].head(10)


In [ ]:
X_train_strength.head(10)


In [ ]:
X_train_final = pd.concat(
    [
        X_train[numerical_features],
        X_train_strength.rename("primary_strength_value"),
        encoded_train_df,
        ingredient_train_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test[numerical_features],
        X_test_strength.rename("primary_strength_value"),
        encoded_test_df,
        ingredient_test_df
    ],
    axis=1
)

X_train_final.shape


In [ ]:
X_train_final.head()


In [ ]:
X_train_final.isnull().sum().sort_values(ascending=False).head(20)


In [ ]:
X_train[["pack_size", "primary_strength"]].describe()


In [ ]:
X_train["primary_strength"].value_counts().head(20)


In [ ]:
pack_size_median = X_train["pack_size"].median()

strength_median = X_train_strength.median()

print("Pack Size Median:", pack_size_median)
print("Strength Median:", strength_median)


 - Replacing Missing values with their respective median

In [ ]:
X_train_final["pack_size"] = X_train_final["pack_size"].fillna(pack_size_median)

X_train_final["primary_strength_value"] = (
    X_train_final["primary_strength_value"]
    .fillna(strength_median)
)

X_test_final["pack_size"] = X_test_final["pack_size"].fillna(pack_size_median)

X_test_final["primary_strength_value"] = (
    X_test_final["primary_strength_value"]
    .fillna(strength_median)
)


In [ ]:
X_test_final.isnull().sum().sum()


 ## Model Training



 A Random Forest Regressor is used as the initial baseline model.



 Random Forest is an ensemble learning algorithm that combines predictions from multiple decision trees, making it robust to noise and capable of learning complex nonlinear relationships between medicine characteristics and price.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

rf_model = RandomForestRegressor(
    n_estimators=20,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_final, y_train)

y_pred_log = rf_model.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test_actual, y_pred)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))

r2 = r2_score(y_test_actual, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")


In [ ]:
print("Train R²:", rf_model.score(X_train_final, y_train))
print("Test R² :", rf_model.score(X_test_final, y_test))


In [ ]:
print(encoded_train_df.shape)
print(len(mlb.classes_))


In [ ]:
from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import time


In [ ]:
lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=127,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)



In [ ]:
start = time.time()

lgb_model.fit(X_train_final, y_train)

end = time.time()

print(f"Training Time: {end-start:.2f} seconds")


In [ ]:
y_pred_log = lgb_model.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)


In [ ]:
mae = mean_absolute_error(y_test_actual, y_pred)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))

r2 = r2_score(y_test_actual, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² : {r2:.4f}")


In [ ]:
print("Train R²:", lgb_model.score(X_train_final, y_train))
print("Test R² :", lgb_model.score(X_test_final, y_test))


In [ ]:
y.describe()


In [ ]:
y.quantile([0.90, 0.95, 0.99, 0.999])


In [ ]:
high_price = clean_df["price_inr"] > 3000

print("Medicines above ₹3000:", high_price.sum())
print("Percentage:", high_price.mean() * 100)
